# Contrail Labeling Helper

Interactive tool for labeling flight contrails.

## Labeling Protocol
- **Blocked**: Flight path obscured (clouds, etc.)
- **Clear**: No contrail visible
- **Dissipate < 10**: Contrail dissipates in < 10 seconds
- **Dissipate > 10**: Contrail dissipates in > 10 seconds
- **Persistent**: Contrail persists for extended time

In [1]:
import pandas as pd
import numpy as np
import cv2
import os
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import json

import utils.adsb_utils as adsb_utils
import utils.projection_utils as proj_utils
from utils.image_data_utils import get_image_data_arizona
# reload import


In [2]:
import importlib
importlib.reload(adsb_utils)
importlib.reload(proj_utils)

<module 'utils.projection_utils' from '/Users/shrenikborad/pless/contrails/utils/projection_utils.py'>

## Configuration
Set your date, camera side, and paths below:

In [11]:
# Configuration
DATE_STR = "2026-01-18"  # Change this to your date
CAMERA_SIDE = "cam2"     # 'east' or 'south'

# Paths
ADSB_CSV_PATH = f"/Users/shrenikborad/pless/easy_adsb/arizona_2026_01_18.csv"
CAMERA_PARAMS_PATH = f"/Users/shrenikborad/pless/contrails/uarizona/cam2/camera_params.json"
BASE_DIR = f'/Users/shrenikborad/pless/contrails/arizona_images/2026-01-18/cam2'
CAMERA_NAME = f"arizona_{CAMERA_SIDE}"

# Output path for labels
LABELS_OUTPUT_PATH = f"./contrail_labels_{DATE_STR}_{CAMERA_NAME}.csv"

print(f"Date: {DATE_STR}")
print(f"Camera: {CAMERA_NAME}")
print(f"Labels will be saved to: {LABELS_OUTPUT_PATH}")

Date: 2026-01-18
Camera: arizona_cam2
Labels will be saved to: ./contrail_labels_2026-01-18_arizona_cam2.csv


## Load Data

In [12]:
print("Projecting to image coordinates...")
intrinsics, distortion, rvec, tvec, origin_gps = proj_utils.load_camera_parameters(CAMERA_PARAMS_PATH)


Projecting to image coordinates...


In [13]:
# Load ADSB data
print("Loading ADSB data...")
df = adsb_utils.read_adsblol_csv(ADSB_CSV_PATH, origin_gps=origin_gps )

# Filter to daytime hours
from_dt = pd.to_datetime(f"{DATE_STR} 06:00:00").tz_localize('America/Phoenix').tz_convert('UTC')
to_dt = pd.to_datetime(f"{DATE_STR} 17:00:00").tz_localize('America/Phoenix').tz_convert('UTC')
df['time'] = pd.to_datetime(df['time'])
df = df[(df['time'] >= from_dt) & (df['time'] < to_dt)]

print(f"Loaded {len(df)} ADSB pings")

# Upsample flight data
print("Upsampling flight data...")
df_upsampled = adsb_utils.get_upsampled_df_for_day(df, max_range_m=100000)



Loading ADSB data...


/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:167: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['flight'] = df.groupby(['icao', 'registration'])['flight'].transform(lambda s: s.ffill().bfill())


Loaded 336631 ADSB pings
Upsampling flight data...


/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['time'] = pd.to_datetime(df['time'])
/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:111: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['alt_gnss_meters'] = df['alt_gnss_meters'].astype(float)
/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:112: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = val

Upsampling all aircraft...
Processing 607 unique aircraft...

Processed 10 aircraft...
Processed 20 aircraft...
Processed 30 aircraft...
Processed 40 aircraft...
Processed 50 aircraft...
Processed 60 aircraft...
Processed 70 aircraft...
Processed 80 aircraft...
Processed 90 aircraft...
Processed 100 aircraft...
Processed 110 aircraft...
Processed 120 aircraft...
Processed 130 aircraft...
Processed 140 aircraft...
Processed 150 aircraft...


...

/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:83: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df[col] = temp_df[col].ffill().infer_objects(copy=False)
/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:83: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df[col] = temp_df[col].ffill().infer_objects(copy=False)
/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:83: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) i

Processed 240 aircraft...
Processed 250 aircraft...
Processed 260 aircraft...
Processed 270 aircraft...
Processed 280 aircraft...
Processed 290 aircraft...
Processed 300 aircraft...
Processed 310 aircraft...
Processed 320 aircraft...


/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:83: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df[col] = temp_df[col].ffill().infer_objects(copy=False)
/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:83: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df[col] = temp_df[col].ffill().infer_objects(copy=False)
/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:83: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) i

Processed 330 aircraft...
Processed 340 aircraft...
Processed 350 aircraft...
Processed 360 aircraft...
Processed 370 aircraft...
Processed 380 aircraft...
Processed 390 aircraft...
Processed 400 aircraft...
Processed 410 aircraft...
Processed 420 aircraft...
Processed 430 aircraft...
Processed 440 aircraft...
Processed 450 aircraft...
Processed 460 aircraft...
Processed 470 aircraft...
Processed 480 aircraft...
Processed 490 aircraft...
Processed 500 aircraft...
Processed 510 aircraft...
Processed 520 aircraft...
Processed 530 aircraft...
Processed 540 aircraft...
Processed 550 aircraft...
Processed 560 aircraft...
Processed 570 aircraft...
Processed 580 aircraft...
Processed 590 aircraft...
Processed 600 aircraft...


/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:139: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_upsampled[['lat', 'lon', 'alt_gnss_meters']].applymap(lambda x: isinstance(x, str)).any(axis=1)


In [14]:
# Load camera parameters and project to image coordinates

image_x, image_y, cam_distance = proj_utils.gps_to_camxy_vasha_fixed(
    df_upsampled['lat'].values,
    df_upsampled['lon'].values,
    df_upsampled['alt_gnss_meters'].values,
    cam_k=intrinsics,
    cam_r=rvec,
    cam_t=tvec,
    camera_gps=origin_gps,
    distortion=distortion
)

df_upsampled['image_x'] = image_x
df_upsampled['image_y'] = image_y
df_upsampled['cam_distance'] = cam_distance

# Load image metadata
print("Loading image metadata...")
image_df = get_image_data_arizona(BASE_DIR)
image_df = image_df[(image_df['time'] >= from_dt) & (image_df['time'] < to_dt)]
max_time = image_df['time'].max() + timedelta(seconds=1)
min_time = image_df['time'].min() - timedelta(seconds=1)
image_df = image_df.sort_values('time').reset_index(drop=True)
df_upsampled = df_upsampled[
    (df_upsampled['time'] >= min_time) & 
    (df_upsampled['time'] <= max_time)
].copy()

print(f"Loaded {len(image_df)} images")
print(f"Time range: {image_df['time'].min()} to {image_df['time'].max()}")

/Users/shrenikborad/pless/contrails/utils/projection_utils.py:33: RuntimeWarning: divide by zero encountered in matmul
  points_cam = (cam_r @ enu_points.T + cam_t).T  # Shape: (N, 3)
/Users/shrenikborad/pless/contrails/utils/projection_utils.py:33: RuntimeWarning: overflow encountered in matmul
  points_cam = (cam_r @ enu_points.T + cam_t).T  # Shape: (N, 3)
/Users/shrenikborad/pless/contrails/utils/projection_utils.py:33: RuntimeWarning: invalid value encountered in matmul
  points_cam = (cam_r @ enu_points.T + cam_t).T  # Shape: (N, 3)


Loading image metadata...
Total 2880
                       time          image_file
0 2026-01-18 07:00:00+00:00  20260118000000.jpg
1 2026-01-18 07:00:30+00:00  20260118000030.jpg
2 2026-01-18 07:01:00+00:00  20260118000100.jpg
3 2026-01-18 07:01:30+00:00  20260118000130.jpg
4 2026-01-18 07:02:00+00:00  20260118000200.jpg
Loaded 1320 images
Time range: 2026-01-18 13:00:00+00:00 to 2026-01-18 23:59:30+00:00


In [15]:
# filter
# 1477.53,609.57,-110.8435989781433,32.38439428076198,2357.407900804103,mountain_right_peak
df_upsampled = df_upsampled[
    ~ ((df_upsampled['image_y'] > 609.57) &
    (df_upsampled['distance_m'] > adsb_utils.haversine_km(32.38439428076198, -110.8435989781433, origin_gps[0], origin_gps[1]) * 1000)
)]

In [16]:
# Get unique flights visible in the time window
# A flight is visible if it has valid image coordinates
image = image_df.iloc[0]
cv2_image = cv2.imread(BASE_DIR + "/"+ image['image_file'])
image_height, image_width = cv2_image.shape[:2]
df_visible = df_upsampled[
    (df_upsampled['image_x'].notna()) & 
    (df_upsampled['image_y'].notna()) &
    (df_upsampled['image_x'] >= 0) &
    (df_upsampled['image_y'] >= 0) &
    (df_upsampled['image_x'] < image_width) &
    (df_upsampled['image_y'] < image_height)
].copy()

# Get flight summary
flight_summary = df_visible.groupby('ident').agg({
    'time': ['min', 'max', 'count'],
    'alt_gnss_meters': ['min', 'max', 'mean'],
    'image_x': 'mean',
    'image_y': 'mean'
}).reset_index()

flight_summary.columns = ['ident', 'time_appear', 'time_disappear', 'n_points', 
                          'alt_min', 'alt_max', 'alt_mean', 'avg_x', 'avg_y']

# Convert altitude to feet for display
flight_summary['alt_min_ft'] = (flight_summary['alt_min'] * 3.28084).round(0)
flight_summary['alt_max_ft'] = (flight_summary['alt_max'] * 3.28084).round(0)

print(f"Found {len(flight_summary)} unique flights visible in frame")
flight_summary.head(10)

Found 339 unique flights visible in frame


,ident,time_appear,time_disappear,n_points,alt_min,alt_max,alt_mean,avg_x,avg_y,alt_min_ft,alt_max_ft
0,A00806,2026-01-18 21:09:10+00:00,2026-01-18 21:14:50+00:00,341,4922.52,7241.03200,6114.398985,1579.645687,579.674000,16150.0,23757.0
1,AAL1024,2026-01-18 18:12:53+00:00,2026-01-18 18:18:53+00:00,305,11026.14,11041.38000,11034.247180,791.962932,562.706821,36175.0,36225.0
2,AAL1048,2026-01-18 16:23:08+00:00,2026-01-18 16:30:51+00:00,354,6492.24,9642.63375,8318.255240,1320.697688,423.304851,21300.0,31636.0
3,AAL1055,2026-01-18 15:27:50+00:00,2026-01-18 15:35:19+00:00,422,10416.54,10424.16000,10421.902891,1473.444470,485.330994,34175.0,34200.0
4,AAL1067,2026-01-18 17:32:39+00:00,2026-01-18 17:39:58+00:00,395,10988.04,10995.66000,10994.043606,1101.887293,530.799883,36050.0,36075.0
5,AAL1083,2026-01-18 21:00:27+00:00,2026-01-18 21:07:40+00:00,331,6492.24,10096.50000,8516.403181,1326.684502,416.938714,21300.0,33125.0
6,AAL1167,2026-01-18 19:06:40+00:00,2026-01-18 19:12:02+00:00,269,10401.30,10424.16000,10412.951399,638.196662,580.066789,34125.0,34200.0
7,AAL1272,2026-01-18 15:07:38+00:00,2026-01-18 15:14:03+00:00,334,11925.30,11940.54000,11928.379940,675.743610,533.672267,39125.0,39175.0
8,AAL1308,2026-01-18 18:53:51+00:00,2026-01-18 18:56:54+00:00,184,10721.34,10721.34000,10721.340000,396.242173,580.309386,35175.0,35175.0
9,AAL1320,2026-01-18 15:52:57+00:00,2026-01-18 15:57:17+00:00,114,6115.05,7666.67250,7025.222834,119.511928,325.673505,20063.0,25153.0


## Initialize Labels DataFrame

In [17]:
# Initialize or load existing labels
LABEL_OPTIONS = ['', 'Blocked', 'Clear', 'Dissipate < 10', 'Dissipate > 10', 'Persistent']

labels_df = flight_summary[['ident', 'time_appear', 'time_disappear', 
                                'alt_min', 'alt_max', 'alt_min_ft', 'alt_max_ft']].copy()
labels_df['alt_appear'] = labels_df['alt_min']
labels_df['alt_disappear'] = labels_df['alt_max']
labels_df['label'] = ''
labels_df['notes'] = ''
labels_df['section'] = 1  # For tracking splits

print(f"Labels dataframe has {len(labels_df)} entries")
labels_df.head()

Labels dataframe has 339 entries


,ident,time_appear,time_disappear,alt_min,alt_max,alt_min_ft,alt_max_ft,alt_appear,alt_disappear,label,notes,section
0,A00806,2026-01-18 21:09:10+00:00,2026-01-18 21:14:50+00:00,4922.52,7241.03200,16150.0,23757.0,4922.52,7241.03200,,,1
1,AAL1024,2026-01-18 18:12:53+00:00,2026-01-18 18:18:53+00:00,11026.14,11041.38000,36175.0,36225.0,11026.14,11041.38000,,,1
2,AAL1048,2026-01-18 16:23:08+00:00,2026-01-18 16:30:51+00:00,6492.24,9642.63375,21300.0,31636.0,6492.24,9642.63375,,,1
3,AAL1055,2026-01-18 15:27:50+00:00,2026-01-18 15:35:19+00:00,10416.54,10424.16000,34175.0,34200.0,10416.54,10424.16000,,,1
4,AAL1067,2026-01-18 17:32:39+00:00,2026-01-18 17:39:58+00:00,10988.04,10995.66000,36050.0,36075.0,10988.04,10995.66000,,,1


## Interactive Labeling Interface

In [18]:
# Export data for HTML labeler
import json

def export_labeling_data(image_df, df_upsampled, labels_df, base_dir, output_json_path):
    """Export all data needed for the HTML labeling interface."""
    
    frames = []
    for idx, row in image_df.iterrows():
        t = row['time']
        
        # Get flights at this time
        flights_at_time = df_upsampled[df_upsampled['time'] == t].copy()
        flights_at_time = flights_at_time[
            (flights_at_time['image_x'].notna()) & 
            (flights_at_time['image_y'].notna()) &
            (flights_at_time['image_x'] >= 0) &
            (flights_at_time['image_y'] >= 0) &
            (flights_at_time['image_x'] < 3000) &  # Filter unrealistic values
            (flights_at_time['image_y'] < 3000)
        ]
        
        flights_list = []
        for _, f in flights_at_time.iterrows():
            flights_list.append({
                'ident': f['ident'],
                'x': float(f['image_x']),
                'y': float(f['image_y']),
                'alt_ft': round(f['alt_gnss_meters'] * 3.28084),
                'heading': float(f['heading']) if 'heading' in f and pd.notna(f['heading']) else 0
            })
        
        frames.append({
            'idx': idx,
            'image_file': row['image_file'],
            'time_utc': t.strftime('%Y-%m-%d %H:%M:%S'),
            'time_local': t.tz_convert('America/Chicago').strftime('%H:%M:%S'),
            'flights': flights_list
        })
    
    # Flight summary for labels
    flights_in_frames = set()
    for frame in frames:
        for flight in frame['flights']:
            flights_in_frames.add(flight['ident'])
    flight_list = []
    for _, row in labels_df.iterrows():
        if row['ident'] not in flights_in_frames:
            continue
        flight_list.append({
            'ident': row['ident'],
            'time_appear': pd.to_datetime(row['time_appear']).strftime('%H:%M:%S'),
            'time_disappear': pd.to_datetime(row['time_disappear']).strftime('%H:%M:%S'),
            'alt_min_ft': int(row['alt_min_ft']),
            'alt_max_ft': int(row['alt_max_ft']),
            'label': row['label'] if pd.notna(row['label']) else '',
            'notes': row['notes'] if pd.notna(row['notes']) else '',
            'section': int(row.get('section', 1))
        })
    
    data = {
        'date': DATE_STR,
        'camera': CAMERA_SIDE,
        'image_base_path': base_dir,
        'total_frames': len(frames),
        'frames': frames,
        'flights': flight_list
    }
    
    with open(output_json_path, 'w') as f:
        json.dump(data, f)
    
    print(f"Exported {len(frames)} frames with flight data to {output_json_path}")
    return data

# Export the data
labeling_data = export_labeling_data(
    image_df, df_upsampled, labels_df, BASE_DIR,
    f"./labeling_data_{DATE_STR}_{CAMERA_NAME}.json"
)
print("Data export complete. You can now use the HTML labeler to label the flights.")

Exported 1320 frames with flight data to ./labeling_data_2026-01-18_arizona_cam2.json
Data export complete. You can now use the HTML labeler to label the flights.
